# Módulo 1 · Etapa 5 — Estrategia de desbalance
**TFM · HAM10000** · Responsable: **Adrián Arias** · MLflow: se añade al run `m1-split-v1` (experimento `m1-datos`)

**Objetivo:** que el modelo no aprenda a responder `nv` a todo (67 % de las imágenes), sin sobrecompensar hacia las clases raras.

**Qué hace**
1. Cuenta las imágenes de cada clase **solo en train** (split v1, verificado por su huella).
2. Calcula dos versiones de pesos por clase:
   - **Pesos puros (`balanced`):** peso inversamente proporcional a la frecuencia. Cada clase aporta lo mismo a la pérdida.
   - **Pesos suavizados (`sqrt`, recomendada):** raíz cuadrada de los anteriores, **normalizada** para que el peso medio por imagen sea 1, igual que con `balanced`.
3. Muestra cuánto aporta cada clase a la pérdida con cada estrategia.
4. Genera `class_weights.json` (el contrato que lee el Módulo 2) y lo añade como artefacto al run del split.

**Por qué normalizar los suavizados:** sin normalizar, la pérdida media baja a ≈ 0,8, lo que equivale a entrenar con una tasa de aprendizaje un 20 % menor.

Entorno: **CPU**, unos 2 minutos. Secretos: `DAGSHUB_USER`, `DAGSHUB_TOKEN`.

## 1. Preparación de la sesión

In [ ]:
AUTOR = "adrian"
OWNER, REPO = "aReinoso007", "tfm-melanoma-cnn-gradcam"
SPLIT_SHA256_ESPERADO = "98485423d758cae1ed8b7adf64c3e92d31a6afd4d48de7a26e89355238e34b38"   # split v1 (scikit-learn 1.9.1)

In [ ]:
import os, json
from pathlib import Path
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/TFM")

REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/{OWNER}/{REPO}.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -r requirements.txt
!pip install -q "scikit-learn==1.9.1"

In [ ]:
import shutil
import pandas as pd, numpy as np
from src.tracking import load_config, file_sha256
cfg = load_config("configs/base.yaml")

SPLIT_DRIVE = DRIVE_ROOT / "data" / "splits" / "split_v1.csv"
assert file_sha256(SPLIT_DRIVE) == SPLIT_SHA256_ESPERADO, "[!] El split de Drive no es el split v1 publicado."
split = pd.read_csv(SPLIT_DRIVE)
print("[+] Split v1 verificado ·", split.split.value_counts().to_dict())

OUT = DRIVE_ROOT / "results" / "m1" / "etapa5"; OUT.mkdir(parents=True, exist_ok=True)
FIG = DRIVE_ROOT / "figures" / "m1" / "etapa5"; FIG.mkdir(parents=True, exist_ok=True)

## 2. Funciones

In [ ]:
# --- Lógica de la Etapa 5 ---
import numpy as np
import pandas as pd

CLASES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]   # orden fijo: índice = etiqueta numérica

def conteos_train(split):
    tr = split[split.split == "train"]
    return tr.dx.value_counts().reindex(CLASES).fillna(0).astype(int)

def pesos(n):
    """Pesos por clase a partir de los conteos de train (Series indexada por clase).

    - balanced: N / (K · n_k). Cada clase aporta lo mismo a la pérdida (1/7).
    - sqrt:     raíz cuadrada de balanced, NORMALIZADA para que el peso medio por imagen
                de train sea 1. Sin normalizar, la pérdida media bajaría (≈ 0,8) y equivaldría
                a entrenar con otra tasa de aprendizaje: la comparación con 'balanced' y con
                'sin pesos' en la ablación no sería justa.
    """
    N, K = int(n.sum()), len(n)
    balanced = N / (K * n)
    raiz = np.sqrt(balanced)
    raiz = raiz * N / float((n * raiz).sum())
    return pd.DataFrame({"n_train": n, "prior_train": n / N, "balanced": balanced, "sqrt": raiz})

def reparto_perdida(tabla):
    """% del peso total de la pérdida que aporta cada clase con cada estrategia."""
    n = tabla.n_train
    out = {"sin_pesos": 100 * n / n.sum()}
    for e in ["balanced", "sqrt"]:
        t = n * tabla[e]
        out[e] = 100 * t / t.sum()
    return pd.DataFrame(out)

def json_pesos(tabla, split_version, split_sha, recomendada="sqrt"):
    r = lambda d: {c: round(float(v), 6) for c, v in d.items()}
    return {
        "split_version": split_version, "split_sha256": split_sha,
        "calculado_con": "solo train", "clases": CLASES,
        "n_train": {c: int(v) for c, v in tabla.n_train.items()},
        "prior_train": r(tabla.prior_train),
        "balanced": r(tabla.balanced),
        "sqrt": r(tabla.sqrt),
        "recomendada": recomendada,
        "keras_class_weight": {str(i): round(float(tabla.loc[c, recomendada]), 6) for i, c in enumerate(CLASES)},
        "notas": [
            "keras_class_weight: claves = índice de clase en el orden de 'clases'. Convertir a int: {int(k): v for k, v in d.items()}.",
            "sqrt está normalizada: el peso medio por imagen de train es 1 (igual que balanced).",
            "Con pesos, las probabilidades del modelo ya no reflejan la frecuencia real de las clases. "
            "Corrección aproximada: p_corregida_k ∝ p_modelo_k / peso_k, renormalizando para que sumen 1.",
            "No combinar con sobremuestreo (compensaría dos veces).",
        ],
    }

In [ ]:
import matplotlib.pyplot as plt
AZUL, TINTA, TINTA2 = "#2a78d6", "#1f1f1d", "#6b6a64"
COLORES = {"sin_pesos": "#b9b8b0", "sqrt": "#2a78d6", "balanced": "#eb6834"}
NOMBRES = {"sin_pesos": "Sin pesos", "sqrt": "Suavizados (raíz)", "balanced": "Puros (balanced)"}
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10, "axes.edgecolor": "#c9c8c1",
                     "axes.labelcolor": TINTA2, "xtick.color": TINTA2, "ytick.color": TINTA2,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e8e7e2", "axes.axisbelow": True, "legend.frameon": False})
def es(x, dec=1):
    return f"{x:,.{dec}f}".replace(",", "X").replace(".", ",").replace("X", ".")
def guardar(fig, nombre):
    fig.savefig(FIG / f"{nombre}.png", bbox_inches="tight", facecolor="white"); plt.show()

## 3. Pesos por clase (solo train)

In [ ]:
n = conteos_train(split)
tabla = pesos(n)

# Verificación cruzada con scikit-learn: 'balanced' debe coincidir exactamente
from sklearn.utils.class_weight import compute_class_weight
tr = split[split.split == "train"]
sk = compute_class_weight("balanced", classes=np.array(CLASES), y=tr.dx.values)
assert np.allclose(sk, tabla.balanced.values), "[!] balanced no coincide con scikit-learn"
print("[+] balanced coincide con compute_class_weight de scikit-learn")

for e in ["balanced", "sqrt"]:
    media = float((tabla.n_train * tabla[e]).sum() / tabla.n_train.sum())
    assert abs(media - 1) < 1e-9
print("[+] Peso medio por imagen de train = 1 en ambas versiones")

vista = tabla.assign(prior_train=100 * tabla.prior_train).round(3)
vista.columns = ["imágenes en train", "% de train", "peso puro", "peso suavizado"]
display(vista)
for e, nom in [("balanced", "puros"), ("sqrt", "suavizados")]:
    print(f"Pesos {nom}: la clase más pesada ({tabla[e].idxmax()}) pesa {tabla[e].max() / tabla[e].min():.1f} veces más que nv")
tabla.to_csv(OUT / "pesos_por_clase.csv")

## 4. Cuánto aporta cada clase a la pérdida
Con cada estrategia, qué parte del peso total de la pérdida corresponde a cada clase. Sin pesos, `nv` domina; con pesos puros, todas aportan lo mismo (14,3 %); los suavizados quedan en medio.

In [ ]:
rep = reparto_perdida(tabla).round(1)
rep.to_csv(OUT / "reparto_perdida.csv"); display(rep)

orden = list(rep.sort_values("sin_pesos").index)          # nv arriba
estr = ["sin_pesos", "sqrt", "balanced"]
alto, y = 0.26, np.arange(len(orden))
fig, ax = plt.subplots(figsize=(8, 4.6))
for j, e in enumerate(estr):
    pos = y + (1 - j) * alto
    v = rep.loc[orden, e].values
    ax.barh(pos, v, height=alto - 0.03, color=COLORES[e], label=NOMBRES[e])
    for p_, val in zip(pos, v):
        ax.text(val + 0.6, p_, es(val), va="center", fontsize=7.5, color=TINTA)
ax.set_yticks(y, orden); ax.grid(axis="y", visible=False)
ax.axvline(100 / 7, color=TINTA2, linewidth=0.8, linestyle=":")
ax.text(100 / 7 + 0.5, len(orden) - 0.45, "reparto igual (14,3 %)", fontsize=8, color=TINTA2)
ax.set_xlabel("% del peso total de la pérdida"); ax.set_xlim(0, 75)
ax.legend(loc="lower right", fontsize=9)
guardar(fig, "fig_reparto_perdida")

## 5. `class_weights.json` (contrato para el Módulo 2)

In [ ]:
SPLIT_SHA = file_sha256(SPLIT_DRIVE)
contrato = json_pesos(tabla, cfg["data"]["split_version"], SPLIT_SHA, recomendada="sqrt")
CW_DRIVE = OUT / "class_weights.json"
CW_DRIVE.write_text(json.dumps(contrato, indent=2, ensure_ascii=False))
CW_REPO = Path("data/splits/class_weights.json")             # va al repo, junto al split
shutil.copy(CW_DRIVE, CW_REPO)

# Comprobación: se lee como lo leerá el Módulo 2
d = json.loads(CW_DRIVE.read_text())
cw = {int(k): v for k, v in d["keras_class_weight"].items()}
assert sorted(cw) == list(range(7)) and d["split_sha256"] == SPLIT_SHA256_ESPERADO
print("[+] class_weights.json en", CW_DRIVE, "y", CW_REPO)
print("Para model.fit(class_weight=...):", {k: round(v, 3) for k, v in cw.items()})
print("class_weights_sha256:", file_sha256(CW_DRIVE))

## 6. Registro en MLflow (en el run del split)
Los pesos se calculan a partir del split, así que se guardan como artefacto del mismo run `m1-split-v1`, como piden las convenciones del repo.

In [ ]:
import mlflow
from src.tracking import setup_mlflow
os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USER")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
setup_mlflow(cfg); mlflow.set_experiment(cfg["mlflow"]["experiments"]["m1"])

runs = mlflow.search_runs(filter_string=f"tags.split_sha256 = '{SPLIT_SHA256_ESPERADO}' and tags.fase = 'split'",
                          output_format="list")
assert len(runs) == 1, f"[!] Se esperaba 1 run del split v1 y hay {len(runs)}."
run_id = runs[0].info.run_id
with mlflow.start_run(run_id=run_id):
    mlflow.log_artifact(str(CW_DRIVE))
    mlflow.log_artifacts(str(OUT), artifact_path="etapa5")
    mlflow.log_artifacts(str(FIG), artifact_path="figuras_etapa5")
    mlflow.set_tags({"class_weighting_recomendada": "sqrt_normalizada",
                     "class_weights_sha256": file_sha256(CW_DRIVE)})
print("[+] Pesos añadidos al run m1-split-v1:", run_id)